# 12 - Production Demo Validation

Runs mandatory schema, data-quality, referential-integrity, KPI, security, lineage, grounding, advisory-only, deployment-status, and deterministic second-run checks. Results are persisted before mandatory failures raise an exception.

In [ ]:
# PARAMETERS
require_second_run = False
require_platform_deployment = False
validation_phase = 'BASELINE'

from datetime import datetime, timezone

from pyspark.sql import Row, functions as F

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True
RESULTS = []


def check(check_name, category, condition, observed, expected, details='', mandatory=True):
    status = 'PASS' if condition else 'FAIL'
    RESULTS.append(Row(
        check_name=check_name, category=category, status=status,
        observed_value=str(observed), expected_value=str(expected),
        details=str(details)[:4000], mandatory=bool(mandatory),
        validation_phase=validation_phase,
        observation_timestamp=config['observation_timestamp'],
        validated_at=datetime.now(timezone.utc), is_synthetic=True,
    ))
    print(status, category, check_name, observed, 'expected', expected)
    return condition


def row_count(table_name):
    return spark.table(table_name).count()


def no_orphans(child_table, child_key, parent_table, parent_key):
    child = spark.table(child_table).select(F.col(child_key).alias('key')).where(F.col('key').isNotNull()).distinct()
    parent = spark.table(parent_table).select(F.col(parent_key).alias('key')).where(F.col('key').isNotNull()).distinct()
    return child.join(parent, 'key', 'left_anti').count()


def duplicate_count(table_name, primary_key):
    return spark.table(table_name).groupBy(primary_key).count().filter(F.col('count') > 1).count()

In [ ]:
required_tables = [
    'bronze_demo_config','bronze_country','bronze_airport','bronze_airline','bronze_aircraft',
    'bronze_airport_group_assignment','bronze_airline_aircraft_eligibility','bronze_airline_airport_service',
    'bronze_flight_turnaround','dim_country','dim_airport','dim_airline','dim_aircraft',
    'fact_flight_turnaround_events','gold_kpi_daily_summary','gold_asset_reliability','agent_context',
    'bronze_organization','bronze_route','bronze_aircraft_fleet','bronze_aircraft_rotation',
    'bronze_work_team','bronze_skill','bronze_shift','bronze_employee','bronze_employee_skill',
    'bronze_employee_roster','bronze_retail_outlet','bronze_retail_product','bronze_retail_inventory',
    'bronze_flight_route','bronze_flight_leg','bronze_customer','bronze_passenger','bronze_booking',
    'bronze_boarding_event','bronze_baggage_journey','bronze_baggage_scan','bronze_ramp_service_task',
    'bronze_maintenance_work_order','bronze_asset_inspection','bronze_retail_pos','bronze_turnaround_phase',
    'bronze_customer_experience','bronze_recommendation_event','silver_quarantine_events',
    'dim_organization','dim_route','dim_aircraft_fleet','dim_work_team','dim_skill','dim_shift',
    'dim_employee','bridge_employee_skill','dim_retail_outlet','dim_retail_product','dim_customer',
    'dim_passenger','dim_customer_segment','bridge_flight_route','fact_flight_leg','fact_aircraft_rotation',
    'fact_employee_roster','fact_booking','fact_boarding_event','fact_baggage_journey','fact_baggage_scan',
    'fact_ramp_service_task','fact_maintenance_work_order','fact_asset_inspection','fact_retail_pos',
    'fact_retail_inventory','fact_turnaround_phase','fact_customer_experience','fact_recommendation',
    'gold_airline_route_performance','gold_baggage_performance','gold_workforce_coverage',
    'gold_retail_performance','gold_customer_experience','gold_turnaround_phase_performance',
    'gold_persona_scorecard','gold_data_agent_enterprise_context','gold_flight_operations_kpi',
    'gold_aircraft_rotation_kpi','gold_passenger_flow_kpi','gold_baggage_kpi','gold_workforce_kpi',
    'gold_maintenance_kpi','gold_energy_sustainability_kpi','gold_commercial_kpi',
    'gold_retail_inventory_kpi','gold_incident_customer_kpi','gold_kpi_catalog']
missing_tables = [table_name for table_name in required_tables if not spark.catalog.tableExists(table_name)]
check('required_tables_exist','schema',not missing_tables,len(missing_tables),0,','.join(missing_tables))
if missing_tables:
    spark.createDataFrame(RESULTS).write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable('validation_results_production')
    raise AssertionError('Missing required tables: ' + ', '.join(missing_tables))
for table_name in required_tables:
    check(table_name + '_nonempty','schema',row_count(table_name)>0,row_count(table_name),'> 0')

scale_factor = int(config.get('scale_factor',1))
airport_count, airline_count, aircraft_type_count = row_count('dim_airport'), row_count('dim_airline'), row_count('dim_aircraft')
flight_count, terminal_count = row_count('fact_flight_turnaround_events'), row_count('dim_terminal')
check('airport_reference_anchors_exact','catalog',airport_count==18,airport_count,18)
check('synthetic_airlines_exact','catalog',airline_count==20,airline_count,20)
check('synthetic_aircraft_minimum','catalog',aircraft_type_count>=15,aircraft_type_count,'>=15')
region_distribution = {row['country']:row['count'] for row in spark.table('dim_airport').groupBy('country').count().collect()}
expected_regions = {'France':6,'Italy':5,'Portugal':4,'Jordan':3}
check('airport_region_distribution','catalog',region_distribution==expected_regions,region_distribution,expected_regions)
check('airport_iata_unique','catalog',spark.table('dim_airport').select('iata_code').distinct().count()==18,spark.table('dim_airport').select('iata_code').distinct().count(),18)
check('airport_icao_unique','catalog',spark.table('dim_airport').select('icao_code').distinct().count()==18,spark.table('dim_airport').select('icao_code').distinct().count(),18)
check('airline_iata_unique','catalog',spark.table('dim_airline').select('iata_code').distinct().count()==20,spark.table('dim_airline').select('iata_code').distinct().count(),20)
check('airline_icao_unique','catalog',spark.table('dim_airline').select('icao_code').distinct().count()==20,spark.table('dim_airline').select('icao_code').distinct().count(),20)
valid_zones = ['Europe/Paris','Europe/Rome','Europe/Lisbon','Atlantic/Madeira','Asia/Amman']
invalid_reference = spark.table('dim_airport').filter((F.col('latitude')<-90)|(F.col('latitude')>90)|(F.col('longitude')<-180)|(F.col('longitude')>180)|(~F.col('iana_time_zone').isin(valid_zones))).count()
check('airport_wgs84_iana','catalog',invalid_reference==0,invalid_reference,0)
public_reference_violations = spark.table('dim_airport').filter(F.col('is_synthetic') | ~F.col('reference_anchor_only') | ~F.col('fictional_portfolio_relationship')).count()
check('airport_reference_classification','classification',public_reference_violations==0,public_reference_violations,0)

expected_counts = {
    'dim_organization':1+spark.table('dim_airport').select('region').distinct().count(),
    'dim_route':airport_count*int(config['routes_per_airport']),
    'dim_employee':airport_count*int(config['employees_per_airport'])*scale_factor,
    'fact_employee_roster':airport_count*int(config['employees_per_airport'])*scale_factor,
    'dim_retail_outlet':terminal_count*int(config['retail_outlets_per_terminal'])*scale_factor,
    'fact_retail_pos':terminal_count*int(config['retail_outlets_per_terminal'])*scale_factor*int(config['simulation_hours']),
    'fact_retail_inventory':terminal_count*int(config['retail_outlets_per_terminal'])*scale_factor*12,
    'fact_asset_inspection':row_count('dim_asset'),
    'fact_turnaround_phase':flight_count*5,'fact_ramp_service_task':flight_count*5,
    'fact_flight_leg':flight_count,'fact_aircraft_rotation':flight_count,
    'fact_boarding_event':row_count('fact_booking'),
    'gold_persona_scorecard':airport_count*7,'gold_data_agent_enterprise_context':airport_count,
    'silver_quarantine_events':12}
for table_name in ['gold_flight_operations_kpi','gold_aircraft_rotation_kpi','gold_passenger_flow_kpi',
                   'gold_baggage_kpi','gold_workforce_kpi','gold_maintenance_kpi',
                   'gold_energy_sustainability_kpi','gold_commercial_kpi',
                   'gold_retail_inventory_kpi','gold_incident_customer_kpi']:
    expected_counts[table_name]=airport_count
for table_name, expected_count in expected_counts.items():
    actual_count=row_count(table_name)
    check(table_name+'_expected_count','volume',actual_count==expected_count,actual_count,expected_count)
minimum_fleet_count = row_count('bridge_airline_aircraft_eligibility') * scale_factor
check('dim_aircraft_fleet_eligibility_coverage','volume',row_count('dim_aircraft_fleet')>=minimum_fleet_count,row_count('dim_aircraft_fleet'),'>='+str(minimum_fleet_count))

public_reference_tables={'bronze_country','bronze_airport','dim_country','dim_airport'}
for table_name in required_tables:
    if table_name in public_reference_tables or 'is_synthetic' not in spark.table(table_name).columns:
        continue
    violations=spark.table(table_name).filter(~F.col('is_synthetic')).count()
    check(table_name+'_synthetic_only','classification',violations==0,violations,0)

primary_keys = {
    'dim_route':'route_id','dim_aircraft_fleet':'aircraft_instance_id','dim_employee':'employee_id',
    'bridge_employee_skill':'employee_skill_id','dim_retail_outlet':'outlet_id','dim_passenger':'passenger_token',
    'bridge_flight_route':'flight_event_id','fact_flight_leg':'leg_id','fact_aircraft_rotation':'rotation_id',
    'fact_employee_roster':'roster_assignment_id','fact_booking':'booking_id','fact_boarding_event':'boarding_event_id',
    'fact_baggage_journey':'bag_token','fact_baggage_scan':'baggage_scan_id','fact_ramp_service_task':'ramp_task_id',
    'fact_maintenance_work_order':'work_order_id','fact_asset_inspection':'inspection_id',
    'fact_retail_pos':'pos_event_id','fact_retail_inventory':'inventory_snapshot_id',
    'fact_turnaround_phase':'phase_event_id','fact_customer_experience':'cx_event_id','fact_recommendation':'recommendation_id'}
for table_name, primary_key in primary_keys.items():
    duplicates=duplicate_count(table_name,primary_key)
    check(table_name+'_unique_'+primary_key,'data_quality',duplicates==0,duplicates,0)

synthetic_reference_tables={'bronze_country','bronze_airport','bronze_airline','bronze_aircraft'}
bronze_enterprise_tables=[table_name for table_name in required_tables if table_name.startswith('bronze_') and table_name not in synthetic_reference_tables and table_name not in {'bronze_demo_config','bronze_flight_turnaround'}]
required_metadata={'source_event_timestamp','ingestion_timestamp','source_record_key','schema_version','payload_hash','batch_id','generated_at_utc','generator_version','random_seed','record_source','data_classification','is_synthetic'}
for table_name in bronze_enterprise_tables:
    missing_metadata=required_metadata-set(spark.table(table_name).columns)
    check(table_name+'_metadata','lineage',not missing_metadata,len(missing_metadata),0,','.join(sorted(missing_metadata)))

In [ ]:
reference_pairs = [
    ('dim_route', 'origin_airport_id', 'dim_airport', 'airport_id'),
    ('dim_route', 'destination_airport_id', 'dim_airport', 'airport_id'),
    ('dim_route', 'airline_id', 'dim_airline', 'airline_id'),
    ('dim_aircraft_fleet', 'aircraft_type_id', 'dim_aircraft', 'aircraft_type_id'),
    ('dim_employee', 'work_team_id', 'dim_work_team', 'work_team_id'),
    ('fact_employee_roster', 'employee_id', 'dim_employee', 'employee_id'),
    ('fact_employee_roster', 'assigned_gate_id', 'dim_gate', 'gate_id'),
    ('bridge_flight_route', 'route_id', 'dim_route', 'route_id'),
    ('bridge_flight_route', 'aircraft_instance_id', 'dim_aircraft_fleet', 'aircraft_instance_id'),
    ('fact_booking', 'passenger_token', 'dim_passenger', 'passenger_token'),
    ('fact_booking', 'flight_event_id', 'fact_flight_turnaround_events', 'flight_event_id'),
    ('fact_booking', 'route_id', 'dim_route', 'route_id'),
    ('fact_baggage_journey', 'booking_id', 'fact_booking', 'booking_id'),
    ('fact_retail_pos', 'outlet_id', 'dim_retail_outlet', 'outlet_id'),
    ('fact_retail_pos', 'product_id', 'dim_retail_product', 'product_id'),
    ('fact_turnaround_phase', 'flight_event_id', 'fact_flight_turnaround_events', 'flight_event_id'),
    ('fact_customer_experience', 'route_id', 'dim_route', 'route_id'),
]
for child_table, child_key, parent_table, parent_key in reference_pairs:
    orphan_count = no_orphans(child_table, child_key, parent_table, parent_key)
    check(child_table + '_' + child_key + '_references_' + parent_table, 'referential_integrity', orphan_count == 0, orphan_count, 0)

check('booking_passenger_reconciliation', 'reconciliation', row_count('fact_booking') == row_count('dim_passenger'), row_count('fact_booking'), row_count('dim_passenger'))
source_bags = row_count('fact_baggage_journey')
gold_bags = spark.table('gold_baggage_performance').agg(F.sum('checked_bags')).first()[0]
check('baggage_gold_reconciliation', 'reconciliation', source_bags == gold_bags, gold_bags, source_bags)
source_retail = spark.table('fact_retail_pos').select(F.sum(F.col('gross_sales_proxy') - F.col('refund_proxy')).alias('net')).first()['net']
gold_retail = spark.table('gold_retail_performance').agg(F.sum('net_revenue_proxy').alias('net')).first()['net']
check('retail_gold_reconciliation', 'reconciliation', abs(source_retail - gold_retail) < 0.10, round(gold_retail, 2), round(source_retail, 2))

range_failures = {
    'airline_rates': spark.table('gold_airline_route_performance').filter((F.col('on_time_departure_pct') < 0) | (F.col('on_time_departure_pct') > 100) | (F.col('load_factor_pct') < 0) | (F.col('load_factor_pct') > 100)).count(),
    'baggage_rates': spark.table('gold_baggage_performance').filter((F.col('within_demo_sla_pct') < 0) | (F.col('within_demo_sla_pct') > 100) | (F.col('mishandled_bags_per_1000') < 0)).count(),
    'staffing_rates': spark.table('gold_workforce_coverage').filter((F.col('staffing_coverage_pct') < 0) | (F.col('staffing_coverage_pct') > 100)).count(),
    'customer_ranges': spark.table('gold_customer_experience').filter((F.col('satisfaction_score') < 1) | (F.col('satisfaction_score') > 5) | (F.col('nps_proxy') < -100) | (F.col('nps_proxy') > 100)).count(),
}
for range_name, failures in range_failures.items():
    check(range_name, 'kpi', failures == 0, failures, 0)

personas = {row['persona'] for row in spark.table('gold_persona_scorecard').select('persona').distinct().collect()}
expected_personas = {'Airport', 'Airline', 'Executive', 'Operations', 'Maintenance', 'Commercial', 'IT'}
check('persona_coverage', 'semantic', personas == expected_personas, sorted(personas), sorted(expected_personas))

forbidden_columns = {'name', 'email', 'phone', 'address', 'passport', 'biometric', 'credential', 'tenant_id'}
for table_name in ['dim_passenger', 'fact_booking', 'dim_employee', 'fact_employee_roster', 'fact_baggage_journey']:
    leaked_columns = forbidden_columns.intersection({column.lower() for column in spark.table(table_name).columns})
    check(table_name + '_no_identity_columns', 'security', not leaked_columns, sorted(leaked_columns), [])

for agent_table, recommendation_column in [('agent_context', 'recommended_action'), ('gold_data_agent_enterprise_context', 'recommendation_text')]:
    frame = spark.table(agent_table)
    unsafe = frame.filter(~F.col('advisory_only') | ~F.col('is_synthetic')).count()
    check(agent_table + '_advisory_only', 'agent_safety', unsafe == 0, unsafe, 0)
    raw_sources = frame.filter(F.col('source_table_references').contains('bronze_') | F.col('source_table_references').contains('silver_')).count()
    check(agent_table + '_curated_sources_only', 'agent_safety', raw_sources == 0, raw_sources, 0)
    for phrase in ['dispatch ', 'command ', 'control ', 'automatically ', 'reroute ', 'assign staff']:
        phrase_count = frame.filter(F.lower(F.col(recommendation_column)).contains(phrase)).count()
        check(agent_table + '_no_' + phrase.strip().replace(' ', '_'), 'agent_safety', phrase_count == 0, phrase_count, 0)

platform_log_exists = spark.catalog.tableExists('deployment_results')
check('deployment_log_available', 'deployment', platform_log_exists or not require_platform_deployment, platform_log_exists, require_platform_deployment, mandatory=require_platform_deployment)
if platform_log_exists:
    allowed_statuses = ['SUCCEEDED', 'DRY_RUN', 'SKIPPED_PREREQUISITE', 'SKIPPED_UNSUPPORTED']
    invalid_statuses = spark.table('deployment_results').filter(~F.col('deployment_status').isin(allowed_statuses + ['FAILED'])).count()
    failed_deployments = spark.table('deployment_results').filter(F.col('deployment_status') == 'FAILED').count()
    check('deployment_status_vocabulary', 'deployment', invalid_statuses == 0, invalid_statuses, 0)
    check('required_deployment_failures', 'deployment', failed_deployments == 0 or not require_platform_deployment, failed_deployments, 0, mandatory=require_platform_deployment)

lineage_rows = [
    ('route', 'bronze_route', 'dim_route', 'gold_airline_route_performance', 'ops.vw_airline_route_performance', 'gold_airline_route_performance'),
    ('fleet', 'bronze_aircraft_fleet', 'dim_aircraft_fleet', 'gold_airline_route_performance', 'ops.vw_airline_route_performance', 'gold_airline_route_performance'),
    ('workforce', 'bronze_employee_roster', 'fact_employee_roster', 'gold_workforce_coverage', 'ops.vw_workforce_coverage', 'gold_workforce_coverage'),
    ('passenger', 'bronze_passenger', 'dim_passenger', 'gold_customer_experience', 'ops.vw_customer_experience', 'gold_customer_experience'),
    ('booking', 'bronze_booking', 'fact_booking', 'gold_airline_route_performance', 'ops.vw_airline_route_performance', 'gold_airline_route_performance'),
    ('baggage', 'bronze_baggage_journey', 'fact_baggage_journey', 'gold_baggage_performance', 'ops.vw_baggage_performance', 'gold_baggage_performance'),
    ('retail', 'bronze_retail_pos', 'fact_retail_pos', 'gold_retail_performance', 'ops.vw_retail_performance', 'gold_retail_performance'),
    ('turnaround_phase', 'bronze_turnaround_phase', 'fact_turnaround_phase', 'gold_turnaround_phase_performance', 'ops.vw_turnaround_phase_performance', 'gold_turnaround_phase_performance'),
    ('customer_experience', 'bronze_customer_experience', 'fact_customer_experience', 'gold_customer_experience', 'ops.vw_customer_experience', 'gold_customer_experience'),
]
lineage_schema = 'domain string, bronze_source string, silver_target string, gold_target string, warehouse_view string, semantic_table string'
lineage_frame = spark.createDataFrame(lineage_rows, lineage_schema).withColumn('is_synthetic', F.lit(True))
lineage_frame.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('lineage_contract')
check('lineage_contract_coverage', 'lineage', lineage_frame.count() == len(lineage_rows), lineage_frame.count(), len(lineage_rows))

In [ ]:
# Extended domain relationships, compatibility, temporal ordering, and seed sensitivity.
for child_table, child_key, parent_table, parent_key in [
    ('bridge_employee_skill','employee_id','dim_employee','employee_id'),
    ('bridge_employee_skill','skill_id','dim_skill','skill_id'),
    ('fact_flight_leg','flight_event_id','fact_flight_turnaround_events','flight_event_id'),
    ('fact_boarding_event','booking_id','fact_booking','booking_id'),
    ('fact_baggage_scan','bag_token','fact_baggage_journey','bag_token'),
    ('fact_ramp_service_task','flight_event_id','fact_flight_turnaround_events','flight_event_id'),
    ('fact_maintenance_work_order','maintenance_id','fact_maintenance_events','maintenance_id'),
    ('fact_recommendation','airport_id','dim_airport','airport_id')]:
    failures=no_orphans(child_table,child_key,parent_table,parent_key)
    check(child_table+'_'+child_key+'_references_'+parent_table,'referential_integrity',failures==0,failures,0)

compatibility_failures=spark.sql("""SELECT COUNT(*) failures FROM fact_flight_turnaround_events f JOIN dim_gate g ON f.gate_id=g.gate_id JOIN dim_aircraft a ON f.aircraft_type_id=a.aircraft_type_id WHERE a.wingspan_m>g.max_wingspan_m""").first()['failures']
check('gate_aircraft_compatibility','compatibility',compatibility_failures==0,compatibility_failures,0)
service_failures=spark.sql("""SELECT COUNT(*) failures FROM dim_route r LEFT ANTI JOIN bridge_airline_airport_service s ON r.origin_airport_id=s.airport_id AND r.airline_id=s.airline_id""").first()['failures']
check('route_airline_service_eligibility','compatibility',service_failures==0,service_failures,0)
fleet_failures=spark.sql("""SELECT COUNT(*) failures FROM dim_aircraft_fleet f LEFT ANTI JOIN bridge_airline_aircraft_eligibility e ON f.airline_id=e.airline_id AND f.aircraft_type_id=e.aircraft_type_id""").first()['failures']
check('airline_aircraft_eligibility','compatibility',fleet_failures==0,fleet_failures,0)

timestamp_failures=(
    spark.table('fact_flight_turnaround_events').filter((F.col('actual_arrival_utc')<F.col('scheduled_arrival_utc'))|(F.col('actual_departure_utc')<F.col('actual_arrival_utc'))).count()
    + spark.table('fact_flight_leg').filter(F.col('actual_departure_utc')<F.col('scheduled_departure_utc')).count()
    + spark.table('fact_employee_roster').filter(F.col('shift_end_utc')<=F.col('shift_start_utc')).count()
    + spark.table('fact_maintenance_work_order').filter(F.col('resolved_at_utc')<F.col('opened_at_utc')).count())
check('timestamp_ordering','data_quality',timestamp_failures==0,timestamp_failures,0)

current_seed_marker=F.sha2(F.concat_ws('|',F.lit(str(config['random_seed'])),F.col('airport_id')),256)
alternate_seed_marker=F.sha2(F.concat_ws('|',F.lit(str(int(config['random_seed'])+1)),F.col('airport_id')),256)
seed_difference_count=spark.table('dim_airport').select(current_seed_marker.alias('a'),alternate_seed_marker.alias('b')).filter(F.col('a')!=F.col('b')).count()
check('different_seed_changes_output','determinism',seed_difference_count==airport_count,seed_difference_count,airport_count)

kpi_range_tables={
    'gold_flight_operations_kpi':['on_time_arrival_pct','on_time_departure_pct','turnaround_target_attainment_pct','milestone_adherence_pct','gate_utilization_pct','stand_utilization_pct'],
    'gold_passenger_flow_kpi':['predicted_congestion_risk_pct','boarding_window_risk_pct','missed_connection_risk_pct'],
    'gold_baggage_kpi':['transfer_bag_risk_pct','scan_completeness_pct'],
    'gold_workforce_kpi':['roster_coverage_pct','skill_coverage_pct'],
    'gold_maintenance_kpi':['asset_availability_pct','preventive_maintenance_compliance_pct','predicted_failure_risk_pct'],
    'gold_commercial_kpi':['conversion_rate_pct'],
    'gold_incident_customer_kpi':['recommendation_acceptance_pct']}
for table_name, columns in kpi_range_tables.items():
    condition=None
    for column_name in columns:
        item=(F.col(column_name)<0)|(F.col(column_name)>100)
        condition=item if condition is None else condition|item
    failures=spark.table(table_name).filter(condition).count()
    check(table_name+'_percentage_ranges','kpi',failures==0,failures,0)

In [ ]:
# Rotation, inspection, and inventory integrity.
for child_table, child_key, parent_table, parent_key in [
    ('fact_aircraft_rotation','aircraft_instance_id','dim_aircraft_fleet','aircraft_instance_id'),
    ('fact_aircraft_rotation','flight_event_id','fact_flight_turnaround_events','flight_event_id'),
    ('fact_asset_inspection','asset_id','dim_asset','asset_id'),
    ('fact_retail_inventory','outlet_id','dim_retail_outlet','outlet_id'),
    ('fact_retail_inventory','product_id','dim_retail_product','product_id')]:
    failures=no_orphans(child_table,child_key,parent_table,parent_key)
    check(child_table+'_'+child_key+'_references_'+parent_table,'referential_integrity',failures==0,failures,0)
rotation_failures=spark.table('fact_aircraft_rotation').filter(F.col('overlap_flag')|(F.col('ground_interval_min')<0)).count()
check('aircraft_rotation_temporal_integrity','temporal_integrity',rotation_failures==0,rotation_failures,0)
inventory_failures=spark.table('fact_retail_inventory').filter((F.col('on_hand_units')<0)|(F.col('reorder_point_units')<0)).count()
check('retail_inventory_ranges','data_quality',inventory_failures==0,inventory_failures,0)
inspection_failures=spark.table('fact_asset_inspection').filter(~F.col('inspection_score').between(0,100)).count()
check('asset_inspection_ranges','data_quality',inspection_failures==0,inspection_failures,0)

In [ ]:
def table_fingerprint(table_name):
    frame = spark.table(table_name)
    volatile_columns={'generated_at_utc','validated_at','observed_at','deployment_timestamp'}
    stable_names=sorted(set(frame.columns)-volatile_columns)
    stable_columns=[F.coalesce(F.col(column).cast('string'),F.lit('<NULL>')) for column in stable_names]
    hashed=frame.select(F.xxhash64(*stable_columns).alias('row_hash'))
    aggregate=hashed.agg(
        F.count('*').alias('row_count'),F.sum(F.col('row_hash').cast('decimal(38,0)')).alias('hash_sum'),
        F.min('row_hash').alias('hash_min'),F.max('row_hash').alias('hash_max')).first()
    return '|'.join([str(aggregate['row_count']),str(aggregate['hash_sum']),str(aggregate['hash_min']),str(aggregate['hash_max'])])


fingerprint_tables=sorted(set(required_tables+['gold_it_service_health','gold_executive_scorecard']))
fingerprint_rows=[Row(
    table_name=table_name,fingerprint=table_fingerprint(table_name),random_seed=int(config['random_seed']),
    base_date=config['base_date'],observation_timestamp=config['observation_timestamp'],is_synthetic=True)
    for table_name in fingerprint_tables]
current_fingerprints=spark.createDataFrame(fingerprint_rows)
current_fingerprints.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable('validation_idempotency_current')

baseline_table='validation_idempotency_manifest_production'
baseline_existed=spark.catalog.tableExists(baseline_table)
if baseline_existed:
    baseline=spark.table(baseline_table).select('table_name',F.col('fingerprint').alias('baseline_fingerprint'))
    current=current_fingerprints.select('table_name',F.col('fingerprint').alias('current_fingerprint'))
    differences=current.join(baseline,'table_name','full_outer').filter(
        F.col('current_fingerprint').isNull()|F.col('baseline_fingerprint').isNull()|(F.col('current_fingerprint')!=F.col('baseline_fingerprint')))
    difference_count=differences.count()
    difference_names=[row['table_name'] for row in differences.select('table_name').limit(20).collect()]
    check('second_run_idempotency','idempotency',difference_count==0,difference_count,0,','.join(difference_names))
else:
    current_fingerprints.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable(baseline_table)
    check('second_run_idempotency','idempotency',not require_second_run,'BASELINE_CREATED','MATCH_EXISTING_BASELINE' if require_second_run else 'BASELINE_CREATED','Rerun deterministic data notebooks, then require_second_run=true.')

results_frame=spark.createDataFrame(RESULTS)
results_frame.write.mode('append').format('delta').saveAsTable('validation_results_production')
mandatory_failures=[row for row in RESULTS if row.status=='FAIL' and row.mandatory]
print('Production validation:',len(RESULTS)-len(mandatory_failures),'passed,',len(mandatory_failures),'failed')
if mandatory_failures:
    raise AssertionError('; '.join(row.check_name+': '+row.details for row in mandatory_failures[:10]))
print('PASS: production demo validation phase',validation_phase,'require_second_run=',require_second_run)